# eSIM 推薦 AI - Fine-tune Qwen2.5-3B
**環境：Google Colab + T4 GPU（免費版即可）**

流程：安裝 → 載模型 → LoRA → 訓練 → 測試 → 輸出 GGUF

In [ ]:
# Cell 1: 安裝套件（約 2 分鐘）
!pip install unsloth
!pip install --upgrade transformers datasets

In [ ]:
# Cell 2: 載入模型（4-bit 量化，T4 大約用 6GB VRAM）
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    dtype=None,          # 自動偵測
    load_in_4bit=True,   # 節省 VRAM
)
print("模型載入完成")

In [ ]:
# Cell 3: 套用 LoRA adapter（只訓練少量參數）
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # LoRA rank，16 是好的起點
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"可訓練參數數量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Cell 4: 上傳並載入資料集
# 先把 alpaca_esim.json 上傳到 Colab（左側檔案面板拖曳）
from datasets import Dataset
import json

with open("alpaca_esim.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Alpaca 格式 prompt 模板
ALPACA_TEMPLATE = """以下是一個關於 eSIM 方案的問題，請給出專業的推薦建議。

### 問題:
{instruction}

### 回答:
{output}"""

def format_sample(sample):
    text = ALPACA_TEMPLATE.format(
        instruction=sample["instruction"],
        output=sample["output"]
    ) + tokenizer.eos_token
    return {"text": text}

dataset = Dataset.from_list(raw_data).map(format_sample)
print(f"資料集大小: {len(dataset)} 筆")
print("\n範例:")
print(dataset[0]["text"][:400])

In [ ]:
# Cell 5: 設定訓練參數並開始訓練（T4 約 15-30 分鐘）
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,           # 訓練 3 輪，可調整
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="./esim_checkpoints",
        save_steps=50,
    ),
)

print("開始訓練...")
trainer_stats = trainer.train()
print(f"訓練完成！耗時: {trainer_stats.metrics['train_runtime']:.0f} 秒")

In [ ]:
# Cell 6: 測試推論
from unsloth.chat_templates import get_chat_template

FastLanguageModel.for_inference(model)

def ask_esim(question: str) -> str:
    prompt = ALPACA_TEMPLATE.format(instruction=question, output="")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### 回答:")[-1].strip()

# 測試問題
test_questions = [
    "我要去日本 7 天，每天用量約中度（社群、導航），預算 NT$500 以內，推薦 eSIM 方案",
    "去歐洲多國旅遊 14 天，不想計算流量，哪個 eSIM 最好？",
    "Airalo 和 Holafly 去韓國哪個比較好？5天，重度用量",
]

for q in test_questions:
    print(f"\n問: {q}")
    print(f"答: {ask_esim(q)}")
    print("-" * 60)

In [ ]:
# Cell 7: 儲存 LoRA adapter（可繼續訓練用）
model.save_pretrained("esim_lora_adapter")
tokenizer.save_pretrained("esim_lora_adapter")
print("LoRA adapter 已儲存")

In [ ]:
# Cell 8: 匯出 GGUF 格式（給 Ollama 用）
# Q4_K_M = 品質與大小的最佳平衡
model.save_pretrained_gguf("esim_model_gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF 檔案已輸出至 esim_model_gguf/")

# 列出輸出檔案
import os
for f in os.listdir("esim_model_gguf"):
    size = os.path.getsize(f"esim_model_gguf/{f}") / 1024 / 1024
    print(f"  {f}  ({size:.1f} MB)")

In [ ]:
# Cell 9: 下載 GGUF 到本機
from google.colab import files
import os

gguf_files = [f for f in os.listdir("esim_model_gguf") if f.endswith(".gguf")]
for f in gguf_files:
    print(f"下載中: {f}")
    files.download(f"esim_model_gguf/{f}")
print("下載完成！接著按照 Modelfile 說明在本機執行 Ollama")

## 下載後在本機執行

1. 把 `.gguf` 檔放進 `Documents/esim_ai/`
2. 確認 `Modelfile` 的 `FROM` 路徑正確
3. 執行：
```bash
ollama create esim-advisor -f Modelfile
ollama run esim-advisor
```